In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict
import random
import os
import cv2
import pandas as pd

seed = 3888
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

img_width, img_height = 100, 100
batch_size = 512
AUTOTUNE = tf.data.AUTOTUNE

tumour_folders = ["Invasive_Tumor", "DCIS_1", "DCIS_2", "Prolif_Invasive_Tumor"]
immune_folders = ["CD4+_T_Cells", "CD8+_T_Cells", "B_Cells", "Stromal"]
sub_label_list = tumour_folders + immune_folders
sub_label_to_index = {name: idx for idx, name in enumerate(sub_label_list)}

def get_file_label_pairs(folders, main_label):
    pairs = []
    for folder in folders:
        files = tf.io.gfile.glob(os.path.join(folder, "*.png"))
        for f in files:
            sub_label = os.path.basename(folder)
            pairs.append((f, main_label, sub_label))
    return pairs

tumour_files = get_file_label_pairs(tumour_folders, main_label=1)
immune_files = get_file_label_pairs(immune_folders, main_label=0)
all_file_label_pairs = tumour_files + immune_files
print(f"Total images before balancing: {len(all_file_label_pairs)}")

def create_balanced_equal_dataset(file_label_pairs, target_size=15000):
    label_to_files = defaultdict(list)
    for f, main_label, sub_label in file_label_pairs:
        label_to_files[sub_label].append((f, main_label, sub_label))

    balanced = []
    for sub_label, files in label_to_files.items():
        originals = files.copy()
        current_size = len(originals)

        if current_size < target_size:
            extra_needed = target_size - current_size
            oversampled_files = random.choices(originals, k=extra_needed)
            balanced.extend((f, m, s, False) for f, m, s in originals)
            balanced.extend((f, m, s, True) for f, m, s in oversampled_files)
        elif current_size > target_size:
            undersampled_files = random.sample(originals, k=target_size)
            balanced.extend((f, m, s, False) for f, m, s in undersampled_files)
        else:
            balanced.extend((f, m, s, False) for f, m, s in originals)

    random.shuffle(balanced)
    return balanced

def get_balanced_unseen_set(file_label_pairs, samples_per_class=25):
    class_to_files = defaultdict(list)
    
    for f, main_label, sub_label in file_label_pairs:
        class_to_files[sub_label].append((f, main_label, sub_label))

    unseen_set = []
    for sub_label, files in class_to_files.items():
        if len(files) < samples_per_class:
            raise ValueError(f"Not enough samples in class '{sub_label}' to select {samples_per_class}")
        selected = random.sample(files, samples_per_class)
        unseen_set.extend((f, m, s, False) for f, m, s in selected)

    random.shuffle(unseen_set)
    return unseen_set

target_size = 15000
unseen_pairs = get_balanced_unseen_set(all_file_label_pairs, samples_per_class=25)
unseen_file_paths = set(f for f, _, _, _ in unseen_pairs)
remaining_pairs = [pair for pair in all_file_label_pairs if pair[0] not in unseen_file_paths]
balanced_pairs = create_balanced_equal_dataset(remaining_pairs, target_size=target_size)
tumour_only_pairs = [pair for pair in balanced_pairs if pair[1] == 1]
print(f"Total images each class balances to: {target_size}")
print(f"Total images after balancing: {len(balanced_pairs)}")

def augment_image(img):
    angles = [0, 90, 180, 270]
    angle = random.choice(angles)
    
    if angle == 90:
        img = tf.image.rot90(img, k=1)
    elif angle == 180:
        img = tf.image.rot90(img, k=2)
    elif angle == 270:
        img = tf.image.rot90(img, k=3)

    flip_type = random.choice(['none', 'horizontal', 'vertical', 'both'])
    if flip_type == 'horizontal':
        img = tf.image.flip_left_right(img)
    elif flip_type == 'vertical':
        img = tf.image.flip_up_down(img)
    elif flip_type == 'both':
        img = tf.image.flip_left_right(img)
        img = tf.image.flip_up_down(img)

    return img

def apply_clahe(img_np):
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    final = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    return final
    
def _load_and_preprocess(path):
    img = cv2.imread(path.numpy().decode('utf-8'))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_width, img_height))
    img = apply_clahe(img)
    img = img.astype(np.float32) / 255.0
    return img
    
def decode_img(img_path, augment=False):
    img = tf.py_function(_load_and_preprocess, [img_path], tf.float32)
    img.set.shape((img_width, img_height, 3))
        
    if augment:
        img = augment_image(img)
    return img

def prepare_dataset(pairs):
    file_paths, main_labels, sub_labels, augment_flags = [], [], [], []

    for f, m, s, a in pairs:
        file_paths.append(f)
        main_labels.append(m)
        sub_labels.append(sub_label_to_index[s] if m==1 else -1)
        augment_flags.append(a)

    file_paths = np.array(file_paths)
    main_labels = np.array(main_labels)
    sub_labels = np.array(sub_labels)
    augment_flags = np.array(augment_flags)

    train_files, test_files, train_main, test_main, train_sub, test_sub, train_aug, test_aug = train_test_split(
        file_paths, main_labels, sub_labels, augment_flags, test_size=0.2, stratify=main_labels, random_state=seed
    )

    def process_path(file_path, main_label, sub_label, augment_flag):
        img = tf.py_function(_load_and_preprocess, [file_path], tf.float32)
        img.set_shape((img_width, img_height, 3))
        
        img = tf.cond(
            augment_flag,
            lambda: augment_image(img),
            lambda: img
        )
    
        return img, {"binary_output": main_label, "multiclass_output": sub_label}

    train_ds = tf.data.Dataset.from_tensor_slices((train_files, train_main, train_sub, train_aug))
    train_ds = train_ds.map(process_path, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(batch_size).prefetch(AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_main, test_sub, test_aug))
    test_ds = test_ds.map(process_path, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    return train_ds, test_ds

def create_unseen_dataset(unseen_pairs):
    file_paths, main_labels, sub_labels, _ = zip(*unseen_pairs)

    file_paths = np.array(file_paths)
    main_labels = np.array(main_labels)
    sub_labels = np.array([sub_label_to_index[s] if m == 1 else -1 for s, m in zip(sub_labels, main_labels)])

    def process_unseen(file_path, main_label, sub_label):
        img = tf.py_function(_load_and_preprocess, [file_path], tf.float32)
        img.set_shape((img_width, img_height, 3))
        return img, {"binary_output": main_label, "multiclass_output": sub_label}

    ds = tf.data.Dataset.from_tensor_slices((file_paths, main_labels, sub_labels))
    ds = ds.map(process_unseen, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)

    return ds

def masked_sparse_categorical_crossentropy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.not_equal(y_true, -1)
    y_true_filtered = tf.boolean_mask(y_true, mask)
    y_pred_filtered = tf.boolean_mask(y_pred, mask)

    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(y_true_filtered, y_pred_filtered)
    )

def build_multitask_model():
    inputs = tf.keras.Input(shape=(img_width, img_height, 3))
    x = tf.keras.layers.Conv2D(32, (3,3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Conv2D(64, (3,3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)

    binary_output = tf.keras.layers.Dense(1, activation='sigmoid', name="binary_output")(x)
    multiclass_output = tf.keras.layers.Dense(8, activation='softmax', name="multiclass_output")(x)

    model = tf.keras.Model(inputs=inputs, outputs=[binary_output, multiclass_output])

    model.compile(
        optimizer='adam',
        loss={
            "binary_output": "binary_crossentropy",
            "multiclass_output": masked_sparse_categorical_crossentropy,
        },
        metrics={
            "binary_output": "accuracy",
            "multiclass_output": "accuracy"
        }
    )

    return model

def train_and_evaluate(train_ds, test_ds, title="Model"):
    model = build_multitask_model()
    model.summary()

    history = model.fit(train_ds, epochs=9, validation_data=test_ds)

    history_df = pd.DataFrame(history.history)

    results = model.evaluate(test_ds)
    print(f"\nTest results - {title}:")
    print(f"Binary Accuracy: {results[3]:.4f}")
    print(f"Multiclass Accuracy: {results[4]:.4f}")

    y_true, y_pred = [], []

    for images, labels in test_ds:
        preds = model.predict(images, verbose=0)
        bin_preds = preds[0].flatten()
        multi_preds = preds[1]

        true_bin = labels['binary_output'].numpy().flatten()
        true_multi = labels['multiclass_output'].numpy().flatten()

        tumour_mask = (true_bin == 1) & (bin_preds > 0.5)
    
        y_true.extend(true_multi[tumour_mask])
        y_pred.extend(np.argmax(multi_preds[tumour_mask], axis=1))
        
    if y_true and y_pred:
        y_true_tumour = np.array(y_true)
        y_pred_tumour = np.array(y_pred)

    return model, y_true_tumour, y_pred_tumour, history_df

train_ds_over, test_ds_over = prepare_dataset(balanced_pairs)
train_ds_tumour, test_ds_tumour = prepare_dataset(tumour_only_pairs)
unseen_ds = create_unseen_dataset(unseen_pairs)

print("\n--- Training with MS+CLAHE ---")
dlcnn_MS_CLAHE, y_true_tumour, y_pred_tumour, history_df = train_and_evaluate(train_ds_over, test_ds_over, title="MS+CLAHE")

Total images before balancing: 124622
Total images each class balances to: 15000
Total images after balancing: 120000

--- Training with MS+CLAHE ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 100, 100, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 98, 98, 32)        │             896 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 49, 49, 32)        │               0 │ conv2d[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 47, 47, 64)        │          18,496 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_1               │ (None, 23, 23, 64)        │               0 │ conv2d_1[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 33856)             │               0 │ max_pooling2d_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 128)               │       4,333,696 │ flatten[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ binary_output (Dense)         │ (None, 1)                 │             129 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multiclass_output (Dense)     │ (None, 8)                 │           1,032 │ dense[0][0]                │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,354,249 (16.61 MB)

 Trainable params: 4,354,249 (16.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/9
188/188 ━━━━━━━━━━━━━━━━━━━━ 225s 1s/step - binary_output_accuracy: 0.7692 - binary_output_loss: 0.4648 - loss: 1.8604 - multiclass_output_accuracy: 0.1684 - multiclass_output_loss: 1.3956 - val_binary_output_accuracy: 0.8805 - val_binary_output_loss: 0.2905 - val_loss: 1.4347 - val_multiclass_output_accuracy: 0.2209 - val_multiclass_output_loss: 1.1443
Epoch 2/9
188/188 ━━━━━━━━━━━━━━━━━━━━ 193s 1s/step - binary_output_accuracy: 0.8856 - binary_output_loss: 0.2904 - loss: 1.3444 - multiclass_output_accuracy: 0.2736 - multiclass_output_loss: 1.0540 - val_binary_output_accuracy: 0.8627 - val_binary_output_loss: 0.3134 - val_loss: 1.3447 - val_multiclass_output_accuracy: 0.2733 - val_multiclass_output_loss: 1.0314
Epoch 3/9
188/188 ━━━━━━━━━━━━━━━━━━━━ 183s 970ms/step - binary_output_accuracy: 0.8933 - binary_output_loss: 0.2706 - loss: 1.2304 - multiclass_output_accuracy: 0.3027 - multiclass_output_loss: 0.9598 - val_binary_output_accuracy: 0.8906 - val_binary_output_loss: 0.2

ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=Image-10/Image10_app/models/dlcnn_MS_CLAHE.

In [19]:
dlcnn_MS_CLAHE.save("Image-10/Image10_app/models/dlcnn_MS_CLAHE.keras")
history_df.to_csv("Image-10/Image10_app/models/history/dlcnn_MS_CLAHE_history.csv", index=False)
np.save("Image-10/Image10_app/models/outputs/dlcnn_MS_CLAHE_ytrue.npy", y_true_tumour)
np.save("Image-10/Image10_app/models/outputs/dlcnn_MS_CLAHE_ypred.npy", y_pred_tumour)

all_confidences = []

for images, labels in test_ds_over:
    preds = dlcnn_MS_CLAHE.predict(images)
    bin_preds = preds[0].flatten()
    multi_preds = preds[1]
    
    true_bin = labels['binary_output'].numpy().flatten()
    
    tumour_mask = (true_bin == 1) & (bin_preds > 0.5)
    
    filtered_multi_preds = multi_preds[tumour_mask]
    confidences = np.max(filtered_multi_preds, axis=1)
    all_confidences.extend(confidences)

np.save("Image-10/Image10_app/models/outputs/dlcnn_MS_CLAHE_conf.npy", np.array(all_confidences))

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
